In [2]:
import pandas as pd
import re

In [3]:
df_ksm_inven = pd.read_csv("../data/raw/ksm_maple_inven_questions.csv")
df_kds_inven = pd.read_csv("../data/raw/kds_maple_inven_question.csv")

display(df_ksm_inven.info())
display(df_kds_inven.info())

display(df_kds_inven['content'].head(10))

<class 'pandas.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   url         650 non-null    str  
 1   category    650 non-null    str  
 2   title       650 non-null    str  
 3   author      473 non-null    str  
 4   created_at  650 non-null    str  
 5   views       650 non-null    int64
 6   likes       650 non-null    int64
 7   content     649 non-null    str  
 8   crawled_at  650 non-null    str  
dtypes: int64(2), str(7)
memory usage: 45.8 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 2743 entries, 0 to 2742
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   post_id     2743 non-null   int64
 1   url         2743 non-null   str  
 2   category    2743 non-null   str  
 3   title       2743 non-null   str  
 4   author      2743 non-null   str  
 5   created_at  2743 non-null   str  
 6   views       2743 non-null   int64
 7   likes       2743 non-null   int64
 8   content     2743 non-null   str  
 9   comments    2743 non-null   str  
 10  crawled_at  2743 non-null   str  
dtypes: int64(3), str(8)
memory usage: 235.9 KB


None

0    [이미지]\n계곡 시즌 끝나가면 헥환 8만 목표로 과금해보려고 하는데,\n과금은 여...
1    목표는 유뇬 8000++인데 테라/메가 버닝 부스터에 챌섭 2자리 정도 남아서 사냥...
2    [이미지]\n이거 뭐부터 손봐야할지 모르겠네여\n목록\n|\n댓글(\n0\n)\n0...
3    부케 소마가 두마리인데 214 200 짜리\n히어로키우고싶어서 근대 케릭터창 자리가...
4    [이미지]\n안녕하세요 메린이 질문드립니다..!\n혹시 이 템은 팔려고하면 얼마정도...
5    8기가램이라 그런가 이거 최적화 방법있나여\n목록\n|\n댓글(\n0\n)\n0\n...
6    [이미지]\n게임 오래할거고 본케입니다.\n템환 6만이구요..\n대적자 에디큡 돌리...
7    [이미지]\n원래눈 왼쪽옵션인데 오른쪽이 떴습니다\n어떤게 더 좋을까요\n목록\n|...
8    복귀 유저라 유챔을 안해봐서 어느 급을 써야 될지 고민인데 혹시 이 셋 중에서 골라...
9    메이플 인게임 자체 필터키 기능을 활성화한 상태에서 shift키에 5~10초 쿨타임...
Name: content, dtype: str

author, post_id, comments, crawled_at, url은 필요없다고 판단

In [4]:
df_kds_inven_cp = df_kds_inven.copy()
df_ksm_inven_cp = df_ksm_inven.copy()

# 분석에 필요없다고 판단한 컬럼 삭제
df_kds_inven_cp = df_kds_inven_cp.drop(columns=['author', 'post_id', 'comments', 'crawled_at', 'url'])
df_ksm_inven_cp = df_ksm_inven_cp.drop(columns=['author', 'crawled_at', 'url'])
# display(df_kds_inven_cp.info())
# display(df_ksm_inven_cp.info())

# 동석님 데이터에서 본문 내용 말고 ui도 함께 크롤링 됨
df_kds_inven_cp["content"] = (
    df_kds_inven_cp["content"]
    .str.split(r"\n목록", n=1, regex=True)
    .str[0]
    .str.strip()
)
display(df_kds_inven_cp)

inven_merged = pd.concat([df_kds_inven_cp, df_ksm_inven_cp])
# print(inven_merged['views'].describe())

inven_merged = inven_merged.dropna(subset=['content'])
# display(inven_merged.shape)
# display(inven_merged.info())

# display(inven_merged['content'].head(10))

# df['created_at'] = pd.to_datetime(df['created_at']).dt.strftime('%Y-%m-%d')

,category,title,created_at,views,likes,content
0,[아이템],[아이템] 헥환 8만 목표 과금 어느정도 해야할까요?,2026-08-14 22:48:00,0,0,"[이미지]\n계곡 시즌 끝나가면 헥환 8만 목표로 과금해보려고 하는데,\n과금은 여..."
1,[직업],[직업] 유뇬작이랑 링크작하려는데,2026-08-14 21:32:00,0,0,목표는 유뇬 8000++인데 테라/메가 버닝 부스터에 챌섭 2자리 정도 남아서 사냥...
2,[아이템],[아이템] 출석 보돌만 하다가 제대로 복귀해보려하는데 템셋 질문좀여,2026-08-14 21:20:00,0,0,[이미지]\n이거 뭐부터 손봐야할지 모르겠네여
3,[직업],[직업] 부케소마 214 200 짜리두마리인데,2026-08-14 21:10:00,0,0,부케 소마가 두마리인데 214 200 짜리\n히어로키우고싶어서 근대 케릭터창 자리가...
4,[아이템],[아이템] 템 가격좀 책정해주실분 계신가요ㅠㅠ,2026-08-14 20:49:00,0,0,[이미지]\n안녕하세요 메린이 질문드립니다..!\n혹시 이 템은 팔려고하면 얼마정도...
...,...,...,...,...,...,...
2738,[기타],[기타] 챌린저스 패스 물품들 본섭리프할때 가져갈수있나요?,2026-07-01 10:50:00,0,0,챌린저스 패스 중에서도\n2만원 과금하면 열수있는 EXP패스 있자나요\n블루베리 농...
2739,[아이템],[아이템] 메린이 템셋 질문,2026-07-01 10:48:00,0,0,[이미지]\n[이미지]\n안녕하십니까 형님들 이번에 메이플 시작한 무자본 뉴비인데 ...
2740,[기타],[기타] 메이플 일반 택배가 12시간이 지났는데도 도착 안할 수 있나요?,2026-07-01 10:43:00,0,0,상대방이 어제 오후 9시에서10시사이에 일반택배를 보냈다고 하고 기다리고 있었는데 ...
2741,[직업],[직업] 환산주스텟 헥사스킬 순서,2026-07-01 10:00:00,0,0,[이미지]\n[이미지]\n현 강화랑 초기화 후랑 차이점이 뭔가요?\n현재 헥사 초기...


In [5]:
# 1. 날짜: YYYY-MM-DD만 유지
inven_merged["created_at"] = inven_merged["created_at"].str[:10]


# 2. category
# 카테고리는 분석 대상 텍스트라기보다 라벨이므로 특수문자 제거
inven_merged["category_clean"] = (
    inven_merged["category"]
    .str.replace(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# 3. title
# 앞에 붙어있는 [아이템], [직업] 등의 카테고리명 제거
# 그 외 특수문자는 NLP를 위해 보존
inven_merged["title_clean"] = (
    inven_merged["title"]
    .str.replace(r"^\[[^\]]+\]\s*", "", regex=True)
    .str.replace(r"\\n|\\r|\\t", " ", regex=True)  # 문자 그대로의 \n, \r, \t
    .str.replace(r"[\n\r\t]", " ", regex=True)     # 실제 개행/탭
    .str.replace(r"\s+", " ", regex=True)           # 연속 공백
    .str.strip()
)


# 4. content
# 특수문자는 최대한 유지
# 개행, 탭, URL 등 명확한 노이즈만 제거
inven_merged["content_clean"] = (
    inven_merged["content"]
    .str.replace(r"^\[[^\]]+\]\s*", "", regex=True)
    .str.replace(r"\\n|\\r|\\t", " ", regex=True)        # 문자 그대로의 \n 등
    .str.replace(r"[\n\r\t]", " ", regex=True)           # 실제 개행/탭
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)  # URL 제거
    .str.replace(r"\s+", " ", regex=True)                 # 연속 공백
    .str.strip()
)

display(inven_merged.shape)
display(inven_merged.info())

sample_inven = inven_merged.copy()

sample_inven = sample_inven.drop(columns=['title', 'category', 'content'])

# 토큰화 전 중간 저장
save_path = "../data/processed/inven_merged_sample.csv"
sample_inven.to_csv(save_path, encoding='utf-8', index=False)


(3392, 9)

<class 'pandas.DataFrame'>
Index: 3392 entries, 0 to 649
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   category        3392 non-null   str   
 1   title           3392 non-null   str   
 2   created_at      3392 non-null   str   
 3   views           3392 non-null   int64 
 4   likes           3392 non-null   int64 
 5   content         3392 non-null   object
 6   category_clean  3392 non-null   str   
 7   title_clean     3392 non-null   str   
 8   content_clean   3392 non-null   object
dtypes: int64(2), object(2), str(5)
memory usage: 265.0+ KB


None

In [6]:
from collections import Counter
from kiwipiepy import Kiwi

kiwi = Kiwi()

def tokenize_text(text):
    tokens = [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith(('NN'))]
    return tokens

sample_inven["title_tokens"] = (
    sample_inven["title_clean"]
    .apply(tokenize_text)
)

sample_inven[
    ["title_clean", "title_tokens"]
].head(10)



,title_clean,title_tokens
0,헥환 8만 목표 과금 어느정도 해야할까요?,"[목표, 과금, 정도]"
1,유뇬작이랑 링크작하려는데,"[유뇬작, 링크, 작]"
2,출석 보돌만 하다가 제대로 복귀해보려하는데 템셋 질문좀여,"[출석, 보, 돌, 복귀, 템, 질문]"
3,부케소마 214 200 짜리두마리인데,"[부케, 소, 마, 짜리, 마리]"
4,템 가격좀 책정해주실분 계신가요ㅠㅠ,"[템, 가격, 책정, 분]"
5,컴터 자꾸 팅겨여,"[컴, 팅]"
6,제네무기 에디 고민 유니크 15 vs 레전 등업,"[무기, 에디, 고민, 유니크, 레, 전, 등, 업]"
7,어떤게 더 좋은가요 직업운 렌입니다,"[것, 직업, 운, 렌]"
8,유챔 하드 세렌급 보조 스펙 질문 있습니다!,"[유채, 하드, 세렌, 급, 보조, 스펙, 질문]"
9,동전꼽기가 운영정책 위반인지 여부,"[동전, 운영, 정책, 위반, 여부]"
